# Production-Style Agentic Memory Architecture

**Goal:** Design an agent that can scale to millions of conversation messages without putting the entire history into the LLM context.

This notebook implements the architecture described in the article:

**User Query → Authorization → Memory Orchestrator → Working / Structured / Archive Memory → Hybrid Retrieval → Reranking → Context Assembly → Agent → Memory Writer → Consolidation**

### Included
- Working memory / sliding window
- Semantic, episodic, procedural, and preference memory
- Full conversation archive
- Hybrid semantic + lexical retrieval
- Recency, importance, entity, temporal, and semantic scoring
- Authorization / tenant isolation before retrieval
- Token-aware context assembly
- Memory writer
- Deduplication and supersession
- Provenance
- Runnable end-to-end demo
- Production deployment notes

The implementation intentionally uses lightweight local components so the notebook runs without external infrastructure. Interfaces are separated so local stores can be replaced with Redis, PostgreSQL/DynamoDB, OpenSearch/Elasticsearch, pgvector/Pinecone/Weaviate, Kafka, etc.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from collections import Counter, defaultdict, deque
import hashlib
import math
import re
import uuid

## 1. Domain model

Every memory carries **identity, authorization scope, temporal metadata, importance, and provenance**.  
That is essential in enterprise systems: retrieval should never occur across an unauthorized namespace.

In [ ]:
class MemoryType(str, Enum):
    WORKING = "working"
    SEMANTIC = "semantic"
    EPISODIC = "episodic"
    PROCEDURAL = "procedural"
    PREFERENCE = "preference"
    ARCHIVE = "archive"


@dataclass(frozen=True)
class SecurityScope:
    tenant_id: str
    user_id: str
    project_id: Optional[str] = None
    persona: Optional[str] = None
    classification: str = "internal"


@dataclass
class Provenance:
    conversation_id: str
    message_ids: List[str]
    created_at: datetime


@dataclass
class MemoryRecord:
    id: str
    text: str
    memory_type: MemoryType
    scope: SecurityScope
    created_at: datetime
    updated_at: datetime
    importance: float = 0.5
    entities: List[str] = field(default_factory=list)
    topic: Optional[str] = None
    confidence: float = 1.0
    valid_from: Optional[datetime] = None
    valid_to: Optional[datetime] = None
    superseded_by: Optional[str] = None
    provenance: Optional[Provenance] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    @property
    def active(self) -> bool:
        return self.superseded_by is None

## 2. Utility functions

For a self-contained notebook, semantic similarity uses a hashed bag-of-words embedding.  
In production, replace `embed()` with your approved embedding model and vector index.

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9_'-]+")

def tokenize(text: str) -> List[str]:
    return [x.lower() for x in TOKEN_RE.findall(text)]

def stable_bucket(token: str, dims: int) -> int:
    digest = hashlib.sha256(token.encode()).digest()
    return int.from_bytes(digest[:8], "big") % dims

def embed(text: str, dims: int = 512) -> List[float]:
    vec = [0.0] * dims
    counts = Counter(tokenize(text))
    for token, count in counts.items():
        vec[stable_bucket(token, dims)] += 1.0 + math.log(count)
    norm = math.sqrt(sum(v*v for v in vec)) or 1.0
    return [v / norm for v in vec]

def cosine(a: Sequence[float], b: Sequence[float]) -> float:
    return sum(x*y for x, y in zip(a, b))

def lexical_similarity(query: str, text: str) -> float:
    q, t = set(tokenize(query)), set(tokenize(text))
    if not q:
        return 0.0
    return len(q & t) / len(q)

def extract_entities(text: str) -> List[str]:
    # Demo heuristic. Production: NER/entity-linking pipeline.
    candidates = re.findall(r"\b(?:[A-Z][A-Za-z0-9_-]*|AWS|Azure|DynamoDB|PostgreSQL|LangGraph)\b", text)
    return sorted(set(candidates))

def utcnow() -> datetime:
    return datetime.now(timezone.utc)

## 3. Storage interfaces

The important production principle is **separation of concerns**.  
The agent should depend on interfaces, not directly on a specific vector database or cache.

In [ ]:
class MemoryStore:
    def __init__(self):
        self.records: Dict[str, MemoryRecord] = {}
        self.vectors: Dict[str, List[float]] = {}

    def put(self, record: MemoryRecord) -> None:
        self.records[record.id] = record
        self.vectors[record.id] = embed(record.text)

    def get(self, memory_id: str) -> Optional[MemoryRecord]:
        return self.records.get(memory_id)

    def authorized_records(self, scope: SecurityScope) -> List[MemoryRecord]:
        # Authorization happens BEFORE retrieval/ranking.
        result = []
        for r in self.records.values():
            if r.scope.tenant_id != scope.tenant_id:
                continue
            if r.scope.user_id != scope.user_id:
                continue
            if scope.project_id and r.scope.project_id not in (None, scope.project_id):
                continue
            if not r.active:
                continue
            result.append(r)
        return result


class WorkingMemory:
    def __init__(self, max_messages: int = 20):
        self.max_messages = max_messages
        self.buffers: Dict[Tuple[str, str], deque] = defaultdict(
            lambda: deque(maxlen=self.max_messages)
        )

    def append(self, scope: SecurityScope, record: MemoryRecord):
        self.buffers[(scope.tenant_id, scope.user_id)].append(record)

    def get(self, scope: SecurityScope) -> List[MemoryRecord]:
        return list(self.buffers[(scope.tenant_id, scope.user_id)])

## 4. Memory Writer

Not every sentence deserves permanent memory.

The writer classifies durable information into:
- semantic facts
- episodes / decisions
- procedures
- preferences

A production implementation can replace these rules with a structured-output LLM classifier plus policy validation.

In [ ]:
class MemoryWriter:
    def __init__(self, store: MemoryStore):
        self.store = store

    def classify(self, text: str) -> Tuple[Optional[MemoryType], float]:
        low = text.lower()

        if any(x in low for x in ["we decided", "approved", "rejected", "selected", "agreed"]):
            return MemoryType.EPISODIC, 0.95

        if any(x in low for x in ["from now on", "always use", "must ", "workflow", "before deploying"]):
            return MemoryType.PROCEDURAL, 0.90

        if any(x in low for x in ["i prefer", "user prefers", "my preference"]):
            return MemoryType.PREFERENCE, 0.80

        if any(x in low for x in [" uses ", " is ", " runs on ", "database", "architecture"]):
            return MemoryType.SEMANTIC, 0.70

        return None, 0.0

    def write(
        self,
        text: str,
        scope: SecurityScope,
        conversation_id: str,
        message_ids: List[str],
        created_at: Optional[datetime] = None,
    ) -> Optional[MemoryRecord]:
        memory_type, importance = self.classify(text)
        if memory_type is None:
            return None

        now = created_at or utcnow()
        record = MemoryRecord(
            id=str(uuid.uuid4()),
            text=text,
            memory_type=memory_type,
            scope=scope,
            created_at=now,
            updated_at=now,
            importance=importance,
            entities=extract_entities(text),
            provenance=Provenance(conversation_id, message_ids, now),
        )
        self.store.put(record)
        return record

## 5. Archive ingestion

The original conversation is retained for provenance and rare-detail retrieval.

In a real system, archive segmentation should use topic/session boundaries rather than blindly slicing every N tokens.

In [ ]:
class ConversationArchive:
    def __init__(self, store: MemoryStore):
        self.store = store

    def add_message(
        self,
        text: str,
        scope: SecurityScope,
        conversation_id: str,
        message_id: str,
        created_at: Optional[datetime] = None,
    ) -> MemoryRecord:
        now = created_at or utcnow()
        record = MemoryRecord(
            id=str(uuid.uuid4()),
            text=text,
            memory_type=MemoryType.ARCHIVE,
            scope=scope,
            created_at=now,
            updated_at=now,
            importance=0.3,
            entities=extract_entities(text),
            provenance=Provenance(conversation_id, [message_id], now),
        )
        self.store.put(record)
        return record

## 6. Hybrid retrieval and ranking

Vector similarity alone is insufficient.

This retriever combines:

**semantic relevance + lexical relevance + recency + importance + entity overlap + temporal match**

Weights can later be learned from retrieval telemetry.

In [ ]:
@dataclass
class RankedMemory:
    record: MemoryRecord
    score: float
    components: Dict[str, float]


class HybridRetriever:
    def __init__(self, store: MemoryStore):
        self.store = store

    @staticmethod
    def recency_score(dt: datetime, half_life_days: float = 90.0) -> float:
        age_days = max(0.0, (utcnow() - dt).total_seconds() / 86400)
        return math.exp(-math.log(2) * age_days / half_life_days)

    def search(
        self,
        query: str,
        scope: SecurityScope,
        top_k: int = 10,
        weights: Optional[Dict[str, float]] = None,
    ) -> List[RankedMemory]:
        weights = weights or {
            "semantic": 0.38,
            "lexical": 0.18,
            "recency": 0.12,
            "importance": 0.20,
            "entity": 0.12,
        }

        qvec = embed(query)
        qentities = set(extract_entities(query))
        ranked = []

        for r in self.store.authorized_records(scope):
            sem = cosine(qvec, self.store.vectors[r.id])
            lex = lexical_similarity(query, r.text)
            rec = self.recency_score(r.created_at)
            imp = r.importance
            entity = (
                len(qentities & set(r.entities)) / max(1, len(qentities))
                if qentities else 0.0
            )

            components = {
                "semantic": sem,
                "lexical": lex,
                "recency": rec,
                "importance": imp,
                "entity": entity,
            }
            score = sum(weights[k] * components[k] for k in weights)
            ranked.append(RankedMemory(r, score, components))

        ranked.sort(key=lambda x: x.score, reverse=True)
        return ranked[:top_k]

## 7. Context Builder

Retrieval and context construction are separate responsibilities.

The context builder enforces a budget and gives recent working memory priority while adding only the best historical memories.

In [ ]:
class ContextBuilder:
    def __init__(self, max_chars: int = 8000):
        self.max_chars = max_chars

    def build(
        self,
        working: List[MemoryRecord],
        retrieved: List[RankedMemory],
    ) -> str:
        sections = []
        used = 0

        def add(label: str, text: str) -> bool:
            nonlocal used
            item = f"[{label}] {text}\n"
            if used + len(item) > self.max_chars:
                return False
            sections.append(item)
            used += len(item)
            return True

        for r in working:
            if not add("RECENT", r.text):
                break

        seen = {r.id for r in working}
        for hit in retrieved:
            if hit.record.id in seen:
                continue
            label = f"{hit.record.memory_type.value.upper()} score={hit.score:.3f}"
            if not add(label, hit.record.text):
                break

        return "".join(sections)

## 8. Memory consolidation

Long-lived agents accumulate duplicates and contradictions.

This demo supports explicit supersession. A production consolidator can additionally:
- cluster near-duplicates
- merge repeated memories
- detect conflicting facts
- maintain validity intervals
- decay low-value memories
- build higher-level episode summaries

In [ ]:
class MemoryConsolidator:
    def __init__(self, store: MemoryStore):
        self.store = store

    def supersede(self, old_id: str, new_id: str) -> None:
        old = self.store.get(old_id)
        new = self.store.get(new_id)
        if not old or not new:
            raise KeyError("Memory not found")
        old.superseded_by = new.id
        old.valid_to = new.created_at
        old.updated_at = utcnow()
        new.valid_from = new.valid_from or new.created_at

    def find_near_duplicates(self, scope: SecurityScope, threshold: float = 0.90):
        records = self.store.authorized_records(scope)
        pairs = []
        for i, a in enumerate(records):
            for b in records[i+1:]:
                sim = cosine(self.store.vectors[a.id], self.store.vectors[b.id])
                if sim >= threshold:
                    pairs.append((a.id, b.id, sim))
        return sorted(pairs, key=lambda x: x[2], reverse=True)

## 9. Memory Orchestrator

This is the main application service. It coordinates:
- archive persistence
- working memory
- long-term extraction
- retrieval
- context assembly

The actual LLM call is deliberately abstracted behind `generate()`.

In [ ]:
class MemoryOrchestrator:
    def __init__(self, working_size: int = 20):
        self.store = MemoryStore()
        self.working = WorkingMemory(working_size)
        self.archive = ConversationArchive(self.store)
        self.writer = MemoryWriter(self.store)
        self.retriever = HybridRetriever(self.store)
        self.context_builder = ContextBuilder()
        self.consolidator = MemoryConsolidator(self.store)

    def ingest(
        self,
        text: str,
        scope: SecurityScope,
        conversation_id: str,
        message_id: str,
        created_at: Optional[datetime] = None,
    ):
        archived = self.archive.add_message(
            text, scope, conversation_id, message_id, created_at
        )

        working_record = MemoryRecord(
            id=f"working-{message_id}",
            text=text,
            memory_type=MemoryType.WORKING,
            scope=scope,
            created_at=archived.created_at,
            updated_at=archived.created_at,
            entities=archived.entities,
            provenance=archived.provenance,
        )
        self.working.append(scope, working_record)

        durable = self.writer.write(
            text, scope, conversation_id, [message_id], archived.created_at
        )
        return archived, durable

    def prepare_context(
        self,
        query: str,
        scope: SecurityScope,
        top_k: int = 8,
    ) -> Tuple[str, List[RankedMemory]]:
        hits = self.retriever.search(query, scope, top_k=top_k)
        context = self.context_builder.build(self.working.get(scope), hits)
        return context, hits

    def generate(self, query: str, context: str) -> str:
        # Replace this adapter with your enterprise LLM gateway / model SDK.
        return (
            "LLM ADAPTER PLACEHOLDER\n\n"
            f"Query: {query}\n\n"
            "The following authorized memory context would be supplied to the model:\n"
            f"{context}"
        )

## 10. End-to-end demo

In [ ]:
memory = MemoryOrchestrator(working_size=4)

scope = SecurityScope(
    tenant_id="acme",
    user_id="user-123",
    project_id="project-x",
    persona="engineering",
    classification="confidential",
)

conversation_id = "conv-architecture"

messages = [
    "Project X initially uses PostgreSQL as the application database.",
    "We discussed DynamoDB because the workload needs predictable low-latency key-value access.",
    "After the architecture review, we decided to use DynamoDB for Project X.",
    "Security approved architecture version 3 after the threat-model review.",
    "From now on, always run the security validation workflow before deploying Project X.",
    "I prefer concise architecture summaries with the decision and rationale first.",
]

for i, message in enumerate(messages, 1):
    memory.ingest(
        text=message,
        scope=scope,
        conversation_id=conversation_id,
        message_id=f"m-{i}",
    )

print(f"Persisted records: {len(memory.store.records)}")
print(f"Working-memory messages: {len(memory.working.get(scope))}")

In [ ]:
query = "What did we finally decide about the database for Project X?"

context, hits = memory.prepare_context(query, scope, top_k=8)

print("TOP RETRIEVED MEMORIES")
print("=" * 80)
for h in hits:
    print(
        f"{h.score:.3f} | {h.record.memory_type.value:10} | "
        f"{h.record.text}"
    )

print("\nASSEMBLED CONTEXT")
print("=" * 80)
print(context)

## 11. Provenance inspection

Every durable memory can point back to its original conversation and message IDs.

In [ ]:
for r in memory.store.authorized_records(scope):
    if r.memory_type == MemoryType.EPISODIC and r.provenance:
        print({
            "memory": r.text,
            "conversation_id": r.provenance.conversation_id,
            "message_ids": r.provenance.message_ids,
            "created_at": r.provenance.created_at.isoformat(),
        })

## 12. Authorization test

A different user in the same tenant must not retrieve the first user's memories.

In [ ]:
other_user_scope = SecurityScope(
    tenant_id="acme",
    user_id="user-999",
    project_id="project-x",
)

unauthorized_hits = memory.retriever.search(
    "What database does Project X use?",
    other_user_scope,
    top_k=10,
)

assert unauthorized_hits == []
print("PASS: cross-user retrieval returned zero memories.")

## 13. Supersession / temporal memory example

Production memory should preserve history instead of silently overwriting facts.

In [ ]:
semantic_records = [
    r for r in memory.store.records.values()
    if r.memory_type == MemoryType.SEMANTIC
]

old_db = next(
    (r for r in semantic_records if "PostgreSQL" in r.text),
    None,
)

new_db = next(
    (r for r in memory.store.records.values()
     if "decided to use DynamoDB" in r.text),
    None,
)

if old_db and new_db:
    memory.consolidator.supersede(old_db.id, new_db.id)
    print("Old memory active:", old_db.active)
    print("Superseded by:", old_db.superseded_by)
    print("New memory active:", new_db.active)
else:
    print("Demo records not found.")

# Production deployment blueprint

The notebook components map cleanly to production infrastructure:

| Notebook component | Production option |
|---|---|
| WorkingMemory | Redis / managed cache |
| ConversationArchive | S3 + DynamoDB/PostgreSQL |
| MemoryStore metadata | PostgreSQL / DynamoDB |
| Vector index | pgvector / OpenSearch / Pinecone / Weaviate |
| HybridRetriever | OpenSearch hybrid search + reranker |
| MemoryWriter | Async LLM extraction service |
| Consolidator | Scheduled/streaming background workers |
| Provenance | Immutable message IDs + audit store |
| Authorization | IAM/RBAC/ABAC enforced before retrieval |
| Events | Kafka / Kinesis / Pub/Sub |
| Observability | OpenTelemetry + traces/metrics |

## Recommended service decomposition

1. **Conversation API** accepts messages and persists immutable events.
2. **Memory Write Service** asynchronously extracts durable memories.
3. **Memory Retrieval Service** performs authorization-aware hybrid search.
4. **Memory Consolidation Worker** deduplicates and supersedes stale memories.
5. **Context Service** assembles a bounded prompt context.
6. **Agent Orchestrator** coordinates model/tool execution.
7. **Audit/Observability layer** records retrieval IDs, scores, provenance, latency, and token usage.

## Scaling to millions of messages

Do not perform linear scans as this local notebook does. At scale:

- Partition by `tenant_id`, `user_id`, and optionally `project_id`.
- Apply metadata/security filters before vector retrieval.
- Use ANN indexes for embeddings.
- Use BM25/full-text indexes for lexical retrieval.
- Rerank only a small candidate set.
- Store raw messages cheaply; promote only durable facts into long-term memory.
- Consolidate asynchronously.
- Cache frequently retrieved memories.
- Keep immutable provenance even when derived memories are superseded.

## Production concerns not to skip

- Encryption at rest and in transit
- Data retention / deletion workflows
- Prompt-injection resistance for retrieved memory
- PII/secrets classification
- Memory poisoning controls
- Tenant isolation
- Idempotent ingestion
- Eventual-consistency handling
- Versioned memory schemas
- Retrieval quality evaluation
- Latency and token budgets
- Human/audit traceability
- Disaster recovery

## Evaluation metrics

Measure memory as its own subsystem:

- Recall@K / Precision@K
- MRR / nDCG
- Decision-memory retrieval accuracy
- Temporal correctness
- Contradiction rate
- Unauthorized retrieval rate (**target: zero**)
- Context utilization
- Retrieval latency P50/P95/P99
- Cost per query
- Memory-write precision
- Provenance coverage

The central design principle remains:

> **A million-message agent is not a million-message prompt. It is an authorization-aware, hierarchical retrieval and memory-management system that supplies the model with the smallest useful context for the current task.**

# Next extensions

For a real implementation, the next engineering step is to replace the local adapters with:

- Redis working memory
- PostgreSQL/pgvector or a managed vector database
- BM25/OpenSearch hybrid retrieval
- an LLM-based structured memory extractor
- a cross-encoder or LLM reranker
- async ingestion via Kafka/Kinesis
- OpenTelemetry tracing
- automated retrieval/evaluation datasets

The architecture in this notebook keeps those integrations behind replaceable interfaces so the agent layer does not need to know which persistence technology is underneath it.